

<img src="images/strathsdr_banner.png" align="left" >

# RFSoC OFDM Transceiver
----

<div class="alert alert-box alert-info">
Please use Jupyter Labs http://board_ip_address/lab for this notebook.
</div>

This notebook demonstrates the implementation of an Orthogonal Frequency Division Multiplexing (OFDM) transceiver on RFSoC. PYNQ is used to control the underlying modulation scheme of the OFDM sub-carriers and for visualisation of data at various stages in the transmit/receive chain, such as the received constellations.       

## Aims 
* To demonstrate a complete OFDM transceiver.
* Explain the various stages OFDM comprises.
* Provide an interactive and responsive means to inspect the data at each stage.

## Table of Contents
* [Introduction](#introduction)
    * [Hardware Setup](#hardware-setup)
    * [Software Setup](#software-setup)
* [Transmit](#transmit)
    * [Symbol Generation](#symbol-gen)
* [Receive](#creating-images)
    * [Constellation Plot](#constellation-plot)
* [Conclusion](#conclusion)

## References
* [Xilinx, Inc, "USP RF Data Converter: LogiCORE IP Product Guide", PG269, v2.4, November 2020](https://www.xilinx.com/support/documentation/ip_documentation/usp_rf_data_converter/v2_4/pg269-rf-data-converter.pdf)

## Revision History
* **v1.0** | 26/02/2021 | OFDM transceiver notebook
* **v1.1** | 30/03/2022 | OFDM DUC and DDC change
* **v1.2** | 17/05/2023 | General notebook for all boards.

----

## Introduction <a class="anchor" id="introduction"></a>

The demonstrator is a complete OFDM transceiver. This notebook will explain each stage of the system with a combination of text, diagrams and live data capture. [Figure 1](#fig-1) below provides an overview of the system.

<a class="anchor" id="fig-1"></a>
<figure>
<img src="./images/system_overview.png" height='75%' width='75%'/>
    <figcaption><b>Figure 1: OFDM Demonstrator System Overview</b></figcaption>
</figure>

The OFDM system starts with generation of random data symbols from 1 of 10 possible modulation schemes (BPSK to 1024 QAM), based on input provided from PYNQ. In accordance with the procedure used in the IEEE 802.11a/g standard, the symbols are grouped into blocks of 48 for mapping to sub-carriers. The OFDM symbol consists of 48 data sub-carriers, 4 pilot sub-carriers and 12 null sub-carriers (including DC). The final OFDM symbol is created by performing a 64 point IFFT and adding a 16 sample Cyclic Prefix (CP). The transmitted signal consists of a continuous stream of OFDM symbols. At the very beginning of the data stream, the L-STF and L-LTF training symbols from the IEEE 802.11a/g standard are transmitted, to aid synchronisation and channel estimation tasks in the receiver. 

In the receiver, timing and frequency synchronisation are performed to acquire symbol timing and correct for any frequency offsets. Once timing is achieved, the FFT is performed to recover the underlying data symbols in each OFDM symbol. The L-LTF symbols are used to estimate the channel frequency response at each sub-carrier position, which subsequently allows the data to be equalised. Finally, the pilot symbols are used to correct for residual phase errors in the phase tracking stage. The recovered symbols are then passed into the PS for visualisation in PYNQ.

### Hardware Setup <a class="anchor" id="hardware-setup"></a>
Your RFSoC development board should be setup in single channel mode with a loopback cable connected between an ADC and DAC.

See the setup instructions [here](01_rfsoc_ofdm_setup.ipynb) for more information.

<div class="alert alert-box alert-danger">
<b>Caution:</b>
    In this demonstration, we generate tones using the RFSoC development board. Your device should be setup in loopback mode. You should understand that the RFSoC platform can also transmit RF signals wirelessly. Remember that unlicensed wireless transmission of RF signals may be illegal in your geographical location. Radio signals may also interfere with nearby devices, such as pacemakers and emergency radio equipment. Note that it is also illegal to intercept and decode particular RF signals. If you are unsure, please seek professional support.
</div>

### Software Setup
The setup for the OFDM demonstration system is nearly complete. The majority of the libraries used by the demonstrator design are contained inside the RFSoC-OFDM software package. We only need to run a few code cells to initialise the environment.

In [1]:
# from pynq import PL
# PL.reset()

In [2]:
from rfsoc_ofdm.overlay import Overlay
# from pynq import Overlay
import ipywidgets as ipw

ofdm_hw = Overlay("/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/notebooks/rfsoc_ofdm_evm9.bit", init_rf_clks=True, clks=409)



/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/notebooks/rfsoc_ofdm_evm9.bit


step 1
hello im here again 
about to return... i hope i have worked
/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm
LMX2594_491.52.txt
got to else...
LMX2594_409.6.txt
/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/xrfclk/lmk04828/LMX2594_409.6.txt
LMK04828_245.76.txt
LMX2594_384.00.txt
.ipynb_checkpoints
/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/xrfclk/lmk04828/LMK04828_245.76.txt
/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/xrfclk/lmk04828/LMX2594_409.6.txt
LMK04828_245.76.txt
LMX2594_409.6.txt
clocks loaded
Configuring adc and dac to: pll_freq=409.6
inside adcs
configure_adcs done
inside dacs
InvSincFIR =  1
mixer mode c2r =  2
configure_dacs done
['InspectorConstellation', 'InspectorConstellation/axi_dma', 'InspectorConstellation/data_inspector_module', 'InspectorEvm', 'InspectorEvm/axi_dma', 'InspectorEvm/axis_switch', 'InspectorEvm/data_inspector_module', 'InspectorReceiver', 'InspectorReceiver/axi_dma', 'InspectorReceiver/axis_switch', 'InspectorRec

In [3]:
mix = (ofdm_hw)
dir(mix)
# (ofdm_hw.dac_block.UpdateEvent(1))

['InspectorConstellation',
 'InspectorConstellation/axi_dma',
 'InspectorConstellation/data_inspector_module',
 'InspectorEvm',
 'InspectorEvm/axi_dma',
 'InspectorEvm/axis_switch',
 'InspectorEvm/data_inspector_module',
 'InspectorReceiver',
 'InspectorReceiver/axi_dma',
 'InspectorReceiver/axis_switch',
 'InspectorReceiver/data_inspector_module',
 'InspectorTransmitter',
 'InspectorTransmitter/axi_dma',
 'InspectorTransmitter/axis_switch',
 'InspectorTransmitter/data_inspector_module',
 'PSDDR',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_deepcopy_dict_from',
 '_ip_map',
 '_register_drivers',
 'adc_block',
 'adc_tile',
 'axi_intc',
 'binfile_nam

----

## Transmit <a class="anchor" id="transmit"></a>

### Symbol Generation <a class="anchor" id="symbol-gen"></a>
There are a total of 10 modulation schemes available to be transmitted. These are generated on the programmable logic and can be chosen between by updating the *ofdm_tx IP core's* **mod** register with a value from 0-9 over AXI4-Lite. [Figure 2](#fig-2) illustrates the IP core as a simplified block diagram.

<a class="anchor" id="fig-2"></a>
<figure>
<img src="./images/symbol_generation.png" height='45%' width='45%'/>
    <figcaption><b>Figure 2: Symbol generation block diagram.</b></figcaption>
</figure>


This drop down widget sends the value associated with each modulation scheme to the ofdm_tx core. Run the cell to use it.

In [4]:
ofdm_hw.ofdm_transmitter.modulation_dropdown.get_widget()

Dropdown(description='Modulation Scheme: ', layout=Layout(width='300px'), options=('BPSK', 'QPSK', '8-PSK', '1…

The output of the symbol generation block has been tapped off, allowing for the live symbols to be visualised in Jupyter Lab. Run the cell below and hit play on the chart to inspect the symbols generated on the programmable logic.

In [5]:
ipw.VBox([ofdm_hw.inspectors['transmitter'].time_plot(),
          ipw.HBox([ofdm_hw.inspectors['transmitter'].channel_widget.get_widget(),
                    
                    ofdm_hw.inspectors['transmitter'].plot_control()])])

    'data': [{'name': 'Real Signal',
              'type': 'scatter',
          …

---

## Receive

### Constellation Plot <a class="anchor" id="constellation-plot"></a>

Run the cell below to see the output of the OFDM receiver displayed as a constellation: 

In [6]:
ipw.VBox([ofdm_hw.inspectors['constellation'].constellation_plot(),
          ofdm_hw.inspectors['constellation'].plot_control()])

    'data': [{'mode': 'markers',
              'type': 'scatter',
              …

In [7]:
# ipw.VBox([ofdm_hw.inspectors['evm'].evm_plot(),
#           ofdm_hw.inspectors['evm'].plot_control()])

In [8]:
ipw.VBox([ofdm_hw.inspectors['receiver'].spectrum_plot(), 
          ipw.HBox([ofdm_hw.inspectors['receiver'].channel_widget.get_widget(),
                    # ofdm_hw.inspectors['receiver'].set_frequency(590),
                    ofdm_hw.inspectors['receiver'].plot_control()])])

    'data': [{'name': 'IQ Spectrum',
              'showlegend': True,
         …

In [9]:
ofdm_hw.ofdm_loopback_application()
# print(ofdm_hw.adc_block.BlockStatus)
# print(ofdm_hw.adc_block.QMCSettings)
# print(ofdm_hw.adc_block.PwrMode)
# # ofdm_hw.adc_tile.Reset()
# dir(ofdm_hw.adc_tile.Reset)

[ 0.15985107+0.13830566j  0.453125  -0.44403076j  0.1552124 -1.04284668j
 -1.05059814+0.43695068j -1.03930664-0.75482178j -0.75891113+1.03051758j
 -0.45336914+0.14282227j -0.74749756-0.1630249j  -0.73828125-1.06414795j
  0.15484619+0.45611572j  0.77154541-1.0479126j   0.14935303+0.15441895j
 -0.73034668-1.04406738j -1.06640625+1.05065918j -1.07873535+1.02441406j
 -0.47741699+1.03198242j -0.76489258+0.72369385j  0.44519043+1.0534668j
  0.43670654+0.75323486j  0.43890381+0.45379639j  0.15551758-1.05212402j
  0.76177979+0.15753174j  0.15856934-0.75628662j  0.75891113+0.15869141j
  0.144104  -0.14886475j -1.03210449-1.06134033j -0.46508789+1.04534912j
 -0.76086426+0.73626709j -1.05096436+0.4331665j  -0.15948486-0.76068115j
 -0.73370361+0.15167236j -0.45635986-0.1585083j  -1.04718018-0.15966797j
 -0.75042725-0.45037842j -0.15454102-0.16009521j -1.05853271-0.16070557j
 -1.05383301-1.06378174j -0.15411377+0.13568115j  0.45013428-0.76556396j
 -1.04547119+0.75189209j -0.13555908+0.44763184j  0.

/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/inspector.py:95: RuntimeWarning:

divide by zero encountered in log10



    'data': [{'name': 'IQ Spectrum',
              …

In [10]:
sw_mmio = ofdm_hw.axis_switch.mmio

CTRL_OFFSET     = 0x000
MI0_MUX_OFFSET  = 0x040
REG_UPDATE_MASK = 0x00000002  # bit 1

def axis_switch_select_input(si_index):
    """
    si_index = 0 -> connect S00 to M00
    si_index = 1 -> connect S01 to M00
    """
    # 1) Program MI0_MUX with desired slave index
    sw_mmio.write(MI0_MUX_OFFSET, si_index & 0xF)

    # 2) Trigger REG_UPDATE so the change takes effect
    ctrl_val = sw_mmio.read(CTRL_OFFSET)
    sw_mmio.write(CTRL_OFFSET, ctrl_val | REG_UPDATE_MASK)

[-31.74108165 -34.37751203 -28.93044647 -32.16077148 -33.71006478
 -42.14419939 -29.59240219 -30.39629481 -39.13482729         -inf
 -42.14419939         -inf -31.74108165 -28.5391893  -35.16107812
 -31.36032426 -31.74108165 -26.7064812  -26.12942835 -29.85218648
 -30.69984085 -32.6017743  -33.71006478 -30.10299957 -36.12359948
 -31.74108165 -30.39629481 -28.72537378 -36.12359948 -42.14419939
 -34.37751203 -29.14047821 -42.14419939 -28.00344687 -30.39629481
 -29.59240219 -31.01438028 -31.36032426 -27.83196588 -28.16479931
 -28.16479931 -26.58117439 -37.40055331 -30.39629481 -31.74108165
 -31.01438028 -39.13482729 -30.69984085 -30.39629481 -31.36032426
 -28.00344687 -35.16107812 -29.85218648 -42.14419939 -42.14419939
 -42.14419939 -33.71006478 -33.71006478 -42.14419939 -35.16107812
 -26.2374632  -30.10299957 -29.59240219 -35.16107812 -34.37751203
 -30.39629481 -27.83196588 -33.71006478 -30.69984085 -31.74108165
 -24.74307454 -33.71006478 -31.36032426 -34.37751203 -27.09362746
 -29.37129

AttributeError: Could not find IP or hierarchy axis_switch in overlay

[ 0.76263428-0.4520874j  -1.03887939-1.06335449j -0.74682617+0.14282227j
  0.44451904-1.04852295j -0.44573975+0.45397949j -0.45715332-0.77374268j
  0.16235352+0.75762939j -0.14886475+1.06359863j  0.45184326+1.04119873j
  1.05932617+0.15124512j  1.04187012+0.15698242j -0.4473877 -0.74707031j
 -0.77191162+1.0456543j   1.05944824+0.14294434j -0.46209717-0.1473999j
  0.16229248-1.07183838j  0.75292969+0.43713379j  0.77081299-1.06549072j
 -0.44537354+1.04504395j  0.45471191+0.44976807j  1.05126953-0.45056152j
  0.76690674-0.44116211j -0.14788818-1.06115723j  1.03729248+0.77026367j
  0.1395874 +0.44787598j -0.46569824-0.76623535j -1.04217529+0.45770264j
  0.16387939-1.04840088j -1.05859375+0.1270752j  -0.7355957 -0.1574707j
 -0.45690918-0.76660156j  0.74188232+1.06488037j -0.45343018+0.15136719j
  0.16363525-0.45452881j  0.76611328-1.05645752j  0.16101074+0.14660645j
  0.14575195-0.45452881j -0.74658203-1.06988525j  0.14929199+0.7456665j
  0.1519165 +0.14978027j  1.06732178-0.7411499j  -1.05

In [ ]:
# Route input 0 -> output
# axis_switch_select_input(1)
ofdm_hw.dac_block.InvSincFIR = 1
print(ofdm_hw.dac_block.InvSincFIR)
dir(ofdm_hw.dac_block.InvSincFIR)

# Later, swap to input 1 -> output
# axis_switch_select_input(1)

In [ ]:
# from rfsoc_ofdm.overlay import Overlay
import numpy as np

# ol = Overlay("rfsoc_ofdm_csv.bit")

player   = ofdm_hw.axis_player          # /axis_player/s_axil
bram_ctl = ofdm_hw.axi_bram_ctrl        # /axi_bram_ctrl/S_AXI  (name may be axi_bram_ctrl_0)
sw_mmio  = ofdm_hw.axis_switch.mmio     # /axis_switch/S_AXI_CTRL

Try changing the modulation scheme using the code cell that was ran earlier. You should be able to visualise the modulation schemes in the plot above.

In [ ]:
csv_path = "/home/xilinx/jupyter_notebooks/rfsoc_ofdm/rfsoc_ofdm/notebooks/DDDDU_5G_id100_ssb120k_pdsch_1_2x_test_20ms.csv"
print("starting loading...")
# Load as float32, shape (N, 2): col 0 = I, col 1 = Q
# iq = np.loadtxt(csv_path, delimiter=',', dtype=np.float32)  # shape (N,2)
print("finished loading...")
fs = 240e6
N=8192
t = np.arange(N) /fs
I = np.sin(2*np.pi*10e6*t)
Q = 0

# Convert to signed Q1.15
scale = 1 << 15
I_fix = np.clip(I, -0.9999695, 0.9999695)
Q_fix = np.clip(Q, -0.9999695, 0.9999695)

I_fix = np.round(I_fix * scale).astype(np.int16)
Q_fix = np.round(Q_fix * scale).astype(np.int16)

# Pack into 32-bit words: [31:16]=I, [15:0]=Q
I32 = I_fix.astype(np.int32)
Q32 = Q_fix.astype(np.int32)
data_words = (I32 << 16) | (Q32 & 0xFFFF)

n_words = len(data_words)
print("Loaded", n_words, "complex samples")

In [ ]:

# Sanity: see how much BRAM MMIO range we have
print("BRAM MMIO length (bytes):", bram_ctl.mmio.length)

for i, word in enumerate(data_words):
    
    bram_ctl.mmio.write(i * 4, int(word))



In [ ]:
player_mmio = player.mmio  # or use player.register_map if you like

CTRL_OFFSET       = 0x00
LENGTH_OFFSET     = 0x04
STATUS_OFFSET     = 0x08
START_ADDR_OFFSET = 0x0C

CTRL_ENABLE = 1 << 0
CTRL_START  = 1 << 1
CTRL_REPEAT = 1 << 2

In [ ]:
def axis_player_start(n_words, start_word=0, repeat=False):
    # 1) Program start address & length
    player_mmio.write(START_ADDR_OFFSET, start_word)
    player_mmio.write(LENGTH_OFFSET,    n_words)

    # 2) Build control word (enable + optional repeat, start=0)
    ctrl = CTRL_ENABLE
    if repeat:
        ctrl |= CTRL_REPEAT

    # Ensure start bit is 0 first
    player_mmio.write(CTRL_OFFSET, ctrl)

    # 3) Now assert start bit (0->1 edge generates start_pulse)
    player_mmio.write(CTRL_OFFSET, ctrl | CTRL_START)

    # (you don't *have* to clear start afterwards; the FSM
    # just looks for the rising edge)

In [ ]:
axis_player_start(n_words, start_word=0, repeat=True)   # continuous loop
# or
axis_player_start(n_words, start_word=0, repeat=False)  # play once

## Conclusion <a class="anchor" id="conclusion"></a>
This notebook has demonstrated a live OFDM transceiver operating on an RFSoC. It has been shown how PYNQ can be used to interact with various parts of the hardware design, offering control of the system and visualisation of data. 

* The various components comprising the OFDM transmitter and receiver have been introduced at a high level. 
    * Modulation symbols were inspected.
    * The received and sychronised constellation were plotted.
* Interacted with a real-time RF system.
    * Changed modulation scheme.

----

⬅️ [Previous Notebook](01_rfsoc_ofdm_setup.ipynb) | [Next Notebook](03_voila_rfsoc_ofdm_demonstrator.ipynb) 🚀

----
----